In [1]:
import json
from pathlib import Path
from collections import Counter

def count_us_images(raw_dir):
    raw_dir    = Path(raw_dir)
    splits_dir = raw_dir / "splits"
    ann_dir    = raw_dir / "annotations"

    # Load all image IDs across all splits
    all_ids = []
    for split_file in splits_dir.glob("*.txt"):
        with open(split_file) as f:
            ids = [l.strip() for l in f if l.strip()]
        all_ids.extend(ids)
        print(f"  {split_file.stem:>5}: {len(ids):,} images")

    print(f"\nTotal image IDs: {len(all_ids):,}")
    print("Scanning annotations for country data...")

    us_ids       = []
    missing      = 0
    no_country   = 0
    country_counts = Counter()

    for i, img_id in enumerate(all_ids):
        if i % 5000 == 0:
            print(f"  Scanned {i:,}/{len(all_ids):,}")

        ann_file = ann_dir / f"{img_id}.json"
        if not ann_file.exists():
            missing += 1
            continue

        with open(ann_file) as f:
            ann = json.load(f)

        country = ann.get("country", None)

        if country is None:
            no_country += 1
            continue

        country_counts[country] += 1

        if country == "US":
            us_ids.append(img_id)

    print(f"\n── Results ─────────────────────────────────────")
    print(f"US images:               {len(us_ids):,}")
    print(f"Missing annotation files:{missing:,}")
    print(f"No country field:        {no_country:,}")
    print(f"\nTop 10 countries:")
    for country, count in country_counts.most_common(10):
        print(f"  {country:>5}: {count:,}")

    return us_ids, country_counts

# Run
us_ids, country_counts = count_us_images("dataset/mtsd_v2_fully_annotated")

   test: 10,544 images
  train: 36,589 images
    val: 5,320 images

Total image IDs: 52,453
Scanning annotations for country data...
  Scanned 0/52,453
  Scanned 5,000/52,453
  Scanned 10,000/52,453
  Scanned 15,000/52,453
  Scanned 20,000/52,453
  Scanned 25,000/52,453
  Scanned 30,000/52,453
  Scanned 35,000/52,453
  Scanned 40,000/52,453
  Scanned 45,000/52,453
  Scanned 50,000/52,453

── Results ─────────────────────────────────────
US images:               0
Missing annotation files:10,544
No country field:        41,909

Top 10 countries:
